In [1]:
import torch
from model.monitors import Plane
from model.retinas import Angular
from model.pixels import Raw, StaticPower, SigmoidPower
from model.perspectives import MonitorRetina, MlpMonitorRetina

In [2]:
device = 'cuda:7'
torch.cuda.set_device(device)

In [3]:

monitor = Plane()
monitor_pixel = Raw()
retina = Angular()
retina_pixel = Raw()

#monitor_retina = MonitorRetina(monitor, monitor_pixel, retina, retina_pixel, 100, 100, [10, 10])
perspective = MlpMonitorRetina(10, 2, 'elu', monitor, monitor_pixel, retina, retina_pixel, 100, 100)

perspective._init(1, 2)
perspective = perspective.to(device)

optimizer = torch.optim.Adam(perspective.parameters(), lr=1e-3)

In [4]:
rays = torch.randn(32, 100, 100, 3).to(device)

plane_out = perspective.monitor.project(rays)

plane_out.shape

torch.Size([32, 100, 100, 2])

In [5]:
parameters = list(perspective.parameters())
for parameter in parameters:
    print(parameter.shape)

torch.Size([3])
torch.Size([3])
torch.Size([3])
torch.Size([3])
torch.Size([10, 2, 1, 1, 1])
torch.Size([1, 10])
torch.Size([10, 10, 1, 1, 1])
torch.Size([1, 10])
torch.Size([1, 10])
torch.Size([3, 10, 1, 1, 1])
torch.Size([1, 3])
torch.Size([1, 3])


In [6]:
perspective.train()
for iter in range(2):
    print(iter)
    stimuli = torch.randn(32, 1, 64, 64).to(device)
    perspectives = torch.randn(32, 2).to(device)

    persp_out = perspective(stimuli, perspectives)

    loss = persp_out.sum()
    loss.backward()
    for name, param in perspective.named_parameters():
        print(f"{name} grad mean: {param.grad.mean().item()} | shape: {param.grad.shape}")
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

0
monitor.center grad mean: -2903.76953125 | shape: torch.Size([3])
monitor.angle grad mean: -2466.4482421875 | shape: torch.Size([3])
monitor.center_std grad mean: 14317.802734375 | shape: torch.Size([3])
monitor.angle_std grad mean: -3632.384521484375 | shape: torch.Size([3])
mlp.linears.0.weights.0 grad mean: -0.00019531250291038305 | shape: torch.Size([10, 2, 1, 1, 1])
mlp.linears.0.biases.0 grad mean: 9.765625145519152e-05 | shape: torch.Size([1, 10])
mlp.linears.1.weights.0 grad mean: 0.0 | shape: torch.Size([10, 10, 1, 1, 1])
mlp.linears.1.gains.0 grad mean: 0.0 | shape: torch.Size([1, 10])
mlp.linears.1.biases.0 grad mean: 0.0003417968691792339 | shape: torch.Size([1, 10])
mlp.linears.2.weights.0 grad mean: 0.0 | shape: torch.Size([3, 10, 1, 1, 1])
mlp.linears.2.gains.0 grad mean: 0.0 | shape: torch.Size([1, 3])
mlp.linears.2.biases.0 grad mean: -2124.3623046875 | shape: torch.Size([1, 3])
1


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.